# Exercise 4 — RiskManager

`RiskManager` combines stop-loss and drawdown limit into a single `filter` call. It also provides `summary` to report how many bars were filtered. This is the interface the trading bot (Days 95–96) will use: one object, one call, before every backtest or live order.

In [ ]:
import pandas as pd, math

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def kelly_fraction(win_rate, avg_win, avg_loss):
    if avg_loss <= 0 or win_rate <= 0 or win_rate >= 1:
        return 0.0
    b = avg_win / avg_loss
    return max(0.0, min(1.0, win_rate - (1 - win_rate) / b))
def is_stopped_out(entry_price, current_price, stop_pct=0.05):
    if entry_price <= 0:
        return False
    return current_price <= entry_price * (1.0 - stop_pct)
def apply_stop_loss(signals, prices, stop_pct=0.05):
    result = signals.copy().astype(float)
    entry_price = None
    for i in range(len(result)):
        if result.iloc[i] == 1:
            if entry_price is None:
                entry_price = float(prices.iloc[i])
            elif is_stopped_out(entry_price, float(prices.iloc[i]), stop_pct):
                result.iloc[i] = 0
                entry_price = None
        else:
            entry_price = None
    return result.astype(int)
def market_drawdown(prices):
    peak = prices.cummax()
    return (prices - peak) / peak

def apply_drawdown_limit(signals, prices, limit=-0.20):
    dd = market_drawdown(prices)
    result = signals.copy().astype(int)
    result[dd < limit] = 0
    return result

class RiskManager:
    """Combined stop-loss + drawdown-limit filter.

    Constructor args:
        stop_pct       : float — stop-loss threshold (default 5%)
        drawdown_limit : float ≤ 0 — market drawdown halt level (default -20%)

    Methods:
        filter(signals, prices)       -> pd.Series {0,1}
        summary(original, filtered)   -> dict
    """

    def __init__(self, stop_pct=0.05, drawdown_limit=-0.20):
        # TODO: store stop_pct and drawdown_limit
        self.stop_pct = stop_pct
        self.drawdown_limit = drawdown_limit

    def filter(self, signals, prices):
        """Apply stop-loss then drawdown limit in sequence.

        Steps:
          1. s = apply_stop_loss(signals, prices, self.stop_pct)
          2. return apply_drawdown_limit(s, prices, self.drawdown_limit)
        """
        # TODO: two lines
        return signals.copy().astype(int)

    def summary(self, original, filtered):
        """Count how many long bars were filtered by risk controls.

        Returns dict with keys:
            total_long_bars, kept_long_bars, filtered_bars, filter_rate
        """
        # TODO:
        # n_orig = int((original == 1).sum())
        # n_kept = int((filtered == 1).sum())
        # return {"total_long_bars": n_orig, "kept_long_bars": n_kept,
        #         "filtered_bars": n_orig-n_kept,
        #         "filter_rate": float((n_orig-n_kept)/max(n_orig,1))}
        return {"total_long_bars": 0, "kept_long_bars": 0,
                "filtered_bars": 0, "filter_rate": 0.0}


### Checks

In [ ]:
checks = 0

# 1 — filter returns same-length Series with {0,1}
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    rm  = RiskManager()
    res = rm.filter(sig, df["Close"])
    assert isinstance(res, pd.Series) and len(res) == len(df)
    assert set(res.unique()).issubset({0, 1})
    checks += 1; print("✅ 1 filter returns same-length Series with {0,1}")
except Exception as e:
    print("❌ 1:", e)

# 2 — filter is at least as conservative as stop-loss alone
try:
    df  = _synthetic()
    sig = pd.Series(1, index=df.index)
    rm  = RiskManager(stop_pct=0.05, drawdown_limit=-0.20)
    sl  = apply_stop_loss(sig, df["Close"], 0.05)
    res = rm.filter(sig, df["Close"])
    assert res.sum() <= sl.sum(),         "combined filter should have ≤ 1s than stop-loss alone"
    checks += 1; print("✅ 2 combined filter is at least as conservative as stop-loss alone")
except Exception as e:
    print("❌ 2:", e)

# 3 — filter: tight limits dramatically reduce exposure
try:
    df    = _synthetic()
    sig   = pd.Series(1, index=df.index)
    rm    = RiskManager(stop_pct=0.02, drawdown_limit=-0.05)
    res   = rm.filter(sig, df["Close"])
    assert res.sum() < sig.sum() * 0.9,         "very tight limits should cut at least 10% of exposure"
    checks += 1; print("✅ 3 tight limits significantly reduce long exposure")
except Exception as e:
    print("❌ 3:", e)

# 4 — summary: correct counts
try:
    df   = _synthetic()
    sig  = pd.Series(1, index=df.index)
    rm   = RiskManager()
    res  = rm.filter(sig, df["Close"])
    info = rm.summary(sig, res)
    assert "total_long_bars" in info and "kept_long_bars" in info
    assert "filtered_bars"   in info and "filter_rate"    in info
    assert info["total_long_bars"] == int(sig.sum())
    assert info["kept_long_bars"]  == int(res.sum())
    assert info["filtered_bars"]   == info["total_long_bars"] - info["kept_long_bars"]
    checks += 1; print("✅ 4 summary counts are consistent")
except Exception as e:
    print("❌ 4:", e)

# 5 — summary filter_rate is between 0 and 1
try:
    df   = _synthetic()
    sig  = pd.Series(1, index=df.index)
    rm   = RiskManager()
    res  = rm.filter(sig, df["Close"])
    info = rm.summary(sig, res)
    assert 0.0 <= info["filter_rate"] <= 1.0,         f"filter_rate out of [0,1]: {info['filter_rate']}"
    print(f"  Filter rate: {info['filter_rate']:.2%} "
          f"({info['filtered_bars']}/{info['total_long_bars']} bars removed)")
    checks += 1; print("✅ 5 filter_rate is in [0.0, 1.0]")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
